In [ ]:
# | default_exp features.fsc_regions

In [ ]:
# | export
from __future__ import annotations
import pandas as pd
import structlog
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()

In [ ]:
# | export
class FSCRegionsEvaluator(FeatureEvaluator):
    """Extracts fragment size category ratios aggregated across gene-level regions.

    Only regions with read coverage (total > 0) are included in aggregation.
    FSC regions are typically ~99.8% covered, so this filter is a consistency
    safeguard rather than a critical fix.
    """

    name = "FSC_regions"
    source_file = ".FSC.regions.parquet"
    tier = 1
    category = "fragmentation"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted

            # Filter to regions with actual read coverage
            if "total" in df.columns:
                n_total = len(df)
                df = df[df["total"] > 0]
                extracted["n_covered_regions"] = len(df)
                extracted["n_total_regions"] = n_total
                if df.empty:
                    log.warning(
                        "no_covered_regions",
                        evaluator=self.name,
                        total_regions=n_total,
                    )
                    return extracted

            cols = set(df.columns)
            ratios = [
                "ultra_short_ratio",
                "core_short_ratio",
                "mono_nucl_ratio",
                "di_nucl_ratio",
                "long_ratio",
            ]

            # Global summary across covered regions
            for r in ratios:
                if r in cols:
                    extracted[f"global_{r}_mean"] = float(df[r].mean())
                    extracted[f"global_{r}_std"] = float(df[r].std())

            # Variance across genes (inter-gene heterogeneity)
            if "gene" in cols:
                for r in ratios:
                    if r in cols:
                        gene_means = df.groupby("gene")[r].mean()
                        if len(gene_means) > 1:
                            extracted[f"{r}_variance_across_genes"] = float(
                                gene_means.std()
                            )

            return extracted
        except Exception as e:
            log.exception("extraction_failed", evaluator=self.name, error=str(e))
            return {}

In [ ]:
# | test
evaluator = FSCRegionsEvaluator()
df_test = pd.DataFrame(
    [
        {"gene": "TP53", "core_short_ratio": 0.2, "long_ratio": 0.05},
        {"gene": "EGFR", "core_short_ratio": 0.4, "long_ratio": 0.01},
    ]
)
res = evaluator.extract(df_test)
assert "global_core_short_ratio_mean" in res
assert "core_short_ratio_variance_across_genes" in res